# CauNagi downstream analysis tutorial

This notebook directly reuses the existing scripts in `Candidate_Regulator_Screening/` and `cascade_classification/`. It does not manually reimplement the five-dimensional evidence extraction.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), Path.cwd().parent) if (path / "Main_code").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the repository root containing Main_code/")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

TEMP_PATH = next(
    (
        path
        for path in (PROJECT_ROOT / "temp_path", PROJECT_ROOT.parent / "temp_path")
        if path.is_dir()
    ),
    PROJECT_ROOT / "temp_path",
)

FINAL_ITERATION = 0  # Change this to the completed CauNagi iteration.
ITERATION_DIR = TEMP_PATH / str(FINAL_ITERATION)
STAGEDATA_DIR = ITERATION_DIR / "stagedata"
DRIVER_RESULTS_DIR = PROJECT_ROOT / "results" / "driver_genes"
CASCADE_RESULTS_DIR = PROJECT_ROOT / "results" / "cascade_classification"

DATASET_PATH = STAGEDATA_DIR / "dataset.h5ad"
ATTRIBUTE_PATH = STAGEDATA_DIR / "attribute.pkl"
IDREM_RESULTS_DIR = ITERATION_DIR / "idremResults"

print(f"Project root: {PROJECT_ROOT}")
print(f"CauNagi output root: {TEMP_PATH}")
print(f"Final iteration: {FINAL_ITERATION}")

## 1. Check the completed CauNagi output

The selected iteration must already contain the merged staged dataset, its attributes, and iDREM results.

In [ ]:
required_paths = {
    "stagedata": STAGEDATA_DIR,
    "dataset.h5ad": DATASET_PATH,
    "attribute.pkl": ATTRIBUTE_PATH,
    "idremResults": IDREM_RESULTS_DIR,
}

for label, path in required_paths.items():
    print(f"{label}: {'OK' if path.exists() else 'MISSING'} -> {path}")

if not DATASET_PATH.is_file():
    raise FileNotFoundError(f"Missing completed dataset: {DATASET_PATH}")

## 2. Generate marker outputs when necessary

This uses the existing public downstream analyst. The results are written to the CauNagi output root because the existing driver-gene script expects `hcmarkers.pkl` and `dynamic_markers.pkl` there.

In [ ]:
from Main_code.get_driver import Analyst

RUN_MARKER_ANALYSIS = False

if RUN_MARKER_ANALYSIS:
    analyst = Analyst(
        data_path=STAGEDATA_DIR,
        iteration=FINAL_ITERATION,
        target_dir=TEMP_PATH,
    )
    analyst.start_analyse(
        progressionmarker_background_sampling=1000,
    )

## 3. Run the existing integrated driver-gene analysis

The existing script collects and integrates the five evidence dimensions internally:

- iterative `geneWeight`;
- iDREM TF evidence;
- dynamic markers;
- regulatory-network evidence;
- hierarchical-clustering markers.

In [ ]:
from Candidate_Regulator_Screening.driver_gene_identification import (
    main as run_driver_gene_analysis,
)

RUN_DRIVER_ANALYSIS = False
PRIOR_NETWORK_PATH = TEMP_PATH / "NicheNet_human.csv"

if RUN_DRIVER_ANALYSIS:
    DRIVER_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    driver_gene_table = run_driver_gene_analysis(
        temp_path=str(TEMP_PATH),
        final_iteration=FINAL_ITERATION,
        prior_net_path=(
            str(PRIOR_NETWORK_PATH)
            if PRIOR_NETWORK_PATH.is_file()
            else None
        ),
        output_dir=str(DRIVER_RESULTS_DIR),
        species="human",
    )
    display(driver_gene_table.head(30))

## 4. Run the existing cell-type-specific driver-gene analysis

This directly calls `Candidate_Regulator_Screening/per_celltype_driver_genes.py`. The current script is designed for a two-state trajectory and writes the cell-type tables into the same driver-results directory used by the cascade script.

In [ ]:
from Candidate_Regulator_Screening import per_celltype_driver_genes

RUN_CELLTYPE_DRIVER_ANALYSIS = False

if RUN_CELLTYPE_DRIVER_ANALYSIS:
    DRIVER_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    celltype_driver_results = per_celltype_driver_genes.main(
        temp_path=str(TEMP_PATH),
        final_iteration=FINAL_ITERATION,
        output_dir=str(DRIVER_RESULTS_DIR),
    )

## 5. Run the existing cascade-classification analysis

The cascade script reads the cell-type driver-gene tables and writes `gene_cascade_classification.csv`. Its input and output directories are set here without changing the existing script.

In [ ]:
import cascade_classification.cascade_classification as cascade

RUN_CASCADE_ANALYSIS = False

if RUN_CASCADE_ANALYSIS:
    required_tables = [
        DRIVER_RESULTS_DIR / f"{cell_type}_driver_genes.csv"
        for cell_type in ["HSPC", "GMP", "Monocyte", "Neutrophil"]
    ]
    missing_tables = [path for path in required_tables if not path.is_file()]
    if missing_tables:
        raise FileNotFoundError(f"Missing driver-gene tables: {missing_tables}")

    CASCADE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    cascade.BASE_DIR = str(DRIVER_RESULTS_DIR)
    cascade.OUT_DIR = str(CASCADE_RESULTS_DIR)
    cascade.main()

## 6. Extract cascade genes

The classification script assigns each gene to a cascade category. This cell exports both all non-`Other` cascade genes and the stricter cascade-core genes.

In [ ]:
classification_path = CASCADE_RESULTS_DIR / "gene_cascade_classification.csv"

if not classification_path.is_file():
    raise FileNotFoundError(
        f"Run the cascade-classification step first: {classification_path}"
    )

classification = pd.read_csv(classification_path)

cascade_categories = [
    "Cascade_Core_Uniform",
    "Cascade_Core_Decay",
    "HSC_Initiator",
    "GMP_Propagator",
    "Mono_Amplifier",
    "Neut_Amplifier",
]
core_categories = [
    "Cascade_Core_Uniform",
    "Cascade_Core_Decay",
]

cascade_genes = (
    classification[
        classification["cascade_category"].isin(cascade_categories)
    ]
    .sort_values(
        ["cascade_category", "HPS", "CTS"],
        ascending=[True, False, False],
    )
    .reset_index(drop=True)
)

cascade_core_genes = (
    classification[
        classification["cascade_category"].isin(core_categories)
    ]
    .sort_values(
        ["HPS", "CTS"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

cascade_genes.to_csv(
    CASCADE_RESULTS_DIR / "cascade_genes.csv",
    index=False,
    encoding="utf-8-sig",
)
cascade_core_genes.to_csv(
    CASCADE_RESULTS_DIR / "cascade_core_genes.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"All cascade genes: {len(cascade_genes)}")
print(f"Cascade-core genes: {len(cascade_core_genes)}")
display(cascade_core_genes.head(30))

## 7. Review generated downstream files

In [ ]:
result_files = sorted(
    path
    for path in {DRIVER_RESULTS_DIR, CASCADE_RESULTS_DIR}
    if path.is_dir()
    for path in path.iterdir()
    if path.is_file()
)

display(
    pd.DataFrame(
        {
            "file": [str(path.relative_to(PROJECT_ROOT)) for path in result_files],
            "size_mb": [
                round(path.stat().st_size / 1024**2, 3)
                for path in result_files
            ],
        }
    )
)